# Parse an ASPECT statistics file

**Author:** Haoyuan Li

ASPECT writes quantities collected by its postprocessors to a `statistics` file. The enabled postprocessors determine the columns, so `AspectStatistics` reads the numbered column descriptions from each file instead of assuming a fixed layout.

In [ ]:
from pathlib import Path

from gdmate.aspect_tools import AspectStatistics

## Load a statistics table

This reduced fixture comes from a real ASPECT model. To analyze another model, replace `statistics_path` with the path to its `output/statistics` file. The repository search below keeps this notebook usable without depending on a particular working directory.

In [ ]:
repository_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "pyproject.toml").is_file()
)
statistics_path = (
    repository_root
    / "tests"
    / "data"
    / "aspect_tools"
    / "statistics_excerpt.txt"
)
statistics = AspectStatistics(statistics_path)
statistics.data.shape

## Inspect and select columns

The underlying `data` object is a pandas `DataFrame`, so standard pandas selection, filtering, and plotting methods are available. Units in trailing parentheses are separated from column names: for example, `Time (years)` becomes the `Time` column, and its unit is available as `statistics.units["Time"]`. A column without a stated unit maps to `None`.

In [ ]:
for number, name in enumerate(statistics.data.columns, start=1):
    print(f"{number:2}: {name} [{statistics.units[name]}]")

statistics.data.head()

In [ ]:
steps = statistics.data["Time step number"]
times = statistics.data["Time"]
timestep_sizes = statistics.data["Time step size"]

list(zip(steps[:5], times[:5], timestep_sizes[:5]))

## Find generated solution files

Filename columns remain strings. ASPECT writes an empty quoted value on timesteps where the corresponding postprocessor did not produce a file, so a Boolean mask selects output timesteps directly.

In [ ]:
has_solution = statistics.data["Visualization file name"] != ""
statistics.data.loc[
    has_solution, ["Time step number", "Time", "Visualization file name"]
]